In [ ]:
# Remove the first # from the next line when installation is required.
%pip install tensorflow matplotlib seaborn scikit-learn certifi

In [ ]:
from pathlib import Path

import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

import tensorflow as tf

from sklearn.metrics import (
    auc,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    fbeta_score,
    jaccard_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_curve,
)

print("TensorFlow version:", tf.__version__)

In [ ]:
# Location of the dataset
BASE_DIR = Path("./animals_dataset")

# Image and training settings
IMAGE_SIZE = 128
BATCH_SIZE = 32
LEARNING_RATE = 0.0001
EPOCHS = 40

# Model and results settings
MODEL_NAME = "VGG19" # This is just to create a folder name
RESULTS_DIR = Path("results") / "multiclass" / MODEL_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", BASE_DIR.resolve())
print("Results folder:", RESULTS_DIR.resolve())

In [ ]:
train_raw = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR / "train",
    labels="inferred",
    label_mode="categorical",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

# Use the same class order for every dataset
class_names = train_raw.class_names

val_raw = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR / "dev",
    labels="inferred",
    label_mode="categorical",
    class_names=class_names,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_raw = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR / "test",
    labels="inferred",
    label_mode="categorical",
    class_names=class_names,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Classes:", class_names)

In [ ]:
for images, labels in train_raw.take(1):
    label_numbers = np.argmax(labels.numpy(), axis=1)

    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("First five class numbers:", label_numbers[:5])
    print("First five class names:", [
        class_names[label] for label in label_numbers[:5]
    ])

In [ ]:
plt.figure(figsize=(10, 8))

for images, labels in train_raw.take(1):
    for index in range(min(9, len(images))):
        plt.subplot(3, 3, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))

        label_number = np.argmax(labels[index].numpy())
        plt.title(class_names[label_number])
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomContrast(0.1),
])

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_raw.map(
    lambda images, labels: (
        data_augmentation(images, training=True),
        labels,
    ),
    num_parallel_calls=AUTOTUNE,
)

val_ds = val_raw

test_ds = test_raw

# Prefetch prepares the next batch while the current batch is being processed
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
vgg19_base = tf.keras.applications.VGG19(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_top=False,
    weights=None,
)

vgg19_base.trainable = True

In [ ]:
NUM_CLASSES = len(class_names)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),
    vgg19_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print("Number of classes:", NUM_CLASSES)
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)

In [ ]:
# Accuracy graph
plt.figure(figsize=(10, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_accuracy.png")
plt.show()

# Loss graph
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_loss.png")
plt.show()

In [ ]:
test_results = model.evaluate(
    test_ds,
    return_dict=True,
)

print("Test results")
print("-" * 30)

for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
# Convert one-hot encoded labels into class numbers
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for _, labels in test_ds
])

# Predict one probability for each class
y_prob = model.predict(test_ds)

# Select the class with the highest probability
y_pred = np.argmax(y_prob, axis=1)

print("Number of test images:", len(y_true))
print("Probability shape:", y_prob.shape)
print("First five predictions:", y_pred[:5])
print("First five predicted classes:", [
    class_names[prediction] for prediction in y_pred[:5]
])

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png")
plt.show()

In [ ]:
# Convert the true class numbers back into one-hot form
y_true_one_hot = tf.keras.utils.to_categorical(
    y_true,
    num_classes=NUM_CLASSES,
)

class_auc_scores = {}

plt.figure(figsize=(10, 7))

for class_index, class_name in enumerate(class_names):
    false_positive_rate, true_positive_rate, _ = roc_curve(
        y_true_one_hot[:, class_index],
        y_prob[:, class_index],
    )

    class_auc = auc(false_positive_rate, true_positive_rate)
    class_auc_scores[class_name] = class_auc

    plt.plot(
        false_positive_rate,
        true_positive_rate,
        label=f"{class_name} (AUC = {class_auc:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Multiclass ROC Curves: One Class Versus the Rest")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roc_curves.png")
plt.show()

macro_auc = np.mean(list(class_auc_scores.values()))

print("AUC for each class")
print("-" * 30)

for class_name, class_auc in class_auc_scores.items():
    print(f"{class_name}: {class_auc:.4f}")

print(f"Macro-average AUC: {macro_auc:.4f}")

In [ ]:
precision, recall, f1_score, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0,
)

cm = confusion_matrix(y_true, y_pred)

class_sensitivities = []
class_specificities = []

for class_index in range(NUM_CLASSES):
    true_positive = cm[class_index, class_index]
    false_negative = cm[class_index, :].sum() - true_positive
    false_positive = cm[:, class_index].sum() - true_positive
    true_negative = cm.sum() - (
        true_positive
        + false_negative
        + false_positive
    )

    sensitivity = (
        true_positive / (true_positive + false_negative)
        if (true_positive + false_negative) > 0
        else 0.0
    )

    specificity = (
        true_negative / (true_negative + false_positive)
        if (true_negative + false_positive) > 0
        else 0.0
    )

    class_sensitivities.append(sensitivity)
    class_specificities.append(specificity)

macro_sensitivity = np.mean(class_sensitivities)
macro_specificity = np.mean(class_specificities)

metrics = {
    "Weighted Precision": precision,
    "Weighted Recall": recall,
    "Macro Sensitivity": macro_sensitivity,
    "Macro Specificity": macro_specificity,
    "Weighted F1-Score": f1_score,
    "Macro-average AUC": macro_auc,
    "Matthews Correlation Coefficient": matthews_corrcoef(
        y_true,
        y_pred,
    ),
    "Cohen's Kappa": cohen_kappa_score(y_true, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(
        y_true,
        y_pred,
    ),
    "Weighted Jaccard Index": jaccard_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    ),
    "Log Loss": log_loss(
        y_true,
        y_prob,
        labels=np.arange(NUM_CLASSES),
    ),
    "Weighted F0.5-Score": fbeta_score(
        y_true,
        y_pred,
        beta=0.5,
        average="weighted",
        zero_division=0,
    ),
}

for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
for images, labels in test_raw.take(1):
    processed_images = preprocess_input(images)
    probabilities = model.predict(
        processed_images,
        verbose=0,
    )

    predictions = np.argmax(probabilities, axis=1)
    true_labels = np.argmax(labels.numpy(), axis=1)

    plt.figure(figsize=(12, 8))

    for index in range(min(9, len(images))):
        true_class = class_names[true_labels[index]]
        predicted_class = class_names[predictions[index]]
        predicted_probability = probabilities[
            index,
            predictions[index],
        ]

        plt.subplot(3, 3, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(
            f"True: {true_class}\n"
            f"Predicted: {predicted_class}\n"
            f"Confidence: {predicted_probability:.2f}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Save the trained model
model_path = RESULTS_DIR / "vgg19_multiclass_classifier.keras"
model.save(model_path)

# Save test metrics
with open(RESULTS_DIR / "testing_metrics.txt", "w", encoding="utf-8") as file:
    for metric_name, metric_value in test_results.items():
        file.write(f"{metric_name}: {metric_value:.4f}\n")

# Save classification metrics
with open(RESULTS_DIR / "classification_metrics.txt", "w", encoding="utf-8") as file:
    for metric_name, metric_value in metrics.items():
        file.write(f"{metric_name}: {metric_value:.4f}\n")

# Save training and validation metrics for every epoch
with open(
    RESULTS_DIR / "training_validation_metrics.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write("Training and Validation Metrics Per Epoch\n")
    file.write("=" * 50 + "\n")

    for epoch_index in range(len(history.history["loss"])):
        file.write(f"Epoch {epoch_index + 1}\n")
        file.write(
            f"  Training Accuracy: "
            f"{history.history['accuracy'][epoch_index]:.4f}\n"
        )
        file.write(
            f"  Validation Accuracy: "
            f"{history.history['val_accuracy'][epoch_index]:.4f}\n"
        )
        file.write(
            f"  Training Loss: "
            f"{history.history['loss'][epoch_index]:.4f}\n"
        )
        file.write(
            f"  Validation Loss: "
            f"{history.history['val_loss'][epoch_index]:.4f}\n"
        )
        file.write("-" * 50 + "\n")

print("Model saved to:", model_path)
print("Other results saved to:", RESULTS_DIR.resolve())